# Dataset Distillation with Optimal Transport & PPDD on Kaggle
This notebook runs the complete **Pretraining -> Distillation (OT/PPDD) -> Evaluation** pipeline with automatic GPU detection (supports both 1x GPU and 2x T4 GPUs).

### Step 1: Environment Setup & Clone Repo

In [ ]:
# 1. Setup working directory and clone repository
import os, shutil, torch

WORKDIR = '/kaggle/working/UROP'
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

!git clone https://github.com/AkmalMohammed-1/UROP.git {WORKDIR}
%cd {WORKDIR}

# 2. Install dependencies
!pip install -q efficientnet_pytorch PyYAML tqdm wandb

# 3. Detect available GPUs
num_gpus = torch.cuda.device_count()
gpu_ids = ','.join(str(i) for i in range(num_gpus))
print(f"\n>>> Successfully detected {num_gpus} GPU(s): {gpu_ids} <<<")

### Step 2: Pretrain 20 Reference Networks

In [ ]:
%cd {WORKDIR}/pretrain

!torchrun --nproc_per_node={num_gpus} --nnodes=1 --master_port=29501 pretrain_script.py \
    --gpu={gpu_ids} \
    --config_path=../config/ipc10/cifar10.yaml

### Step 3: Run Distillation with Optimal Transport

In [ ]:
%cd {WORKDIR}/condense

!torchrun --nproc_per_node={num_gpus} --nnodes=1 --master_port=29502 condense_script.py \
    --gpu={gpu_ids} \
    --ipc=10 \
    --config_path=../config/ipc10/cifar10.yaml

### Step 4: Automatically Find Checkpoint & Run Evaluation

In [ ]:
import glob, os

# Search for generated distilled dataset checkpoint
checkpoints = glob.glob(f"{WORKDIR}/results/**/data_20000.pt", recursive=True)
if not checkpoints:
    checkpoints = glob.glob(f"{WORKDIR}/results/**/*.pt", recursive=True)

assert len(checkpoints) > 0, "Error: Distilled checkpoint not found in results directory!"
latest_checkpoint = sorted(checkpoints, key=os.path.getmtime)[-1]
print(f"\n>>> Evaluating distilled dataset: {latest_checkpoint} <<<")

%cd {WORKDIR}/evaluation

!torchrun --nproc_per_node=1 --nnodes=1 --master_port=29503 evaluation_script.py \
    --gpu=0 \
    --ipc=10 \
    --config_path=../config/ipc10/cifar10.yaml \
    --load_path="{latest_checkpoint}"